In [27]:
# Example usage of load_pretrained_model for loading a pretrained 3D ResNet model

import torch
from resnet3d.model import generate_model, load_pretrained_model

model_depth = 34  # Example depth, can be 18, 34, 50, etc.

# Define options as an object with required attributes
class Opt:
    model = 'resnet'
    model_depth = model_depth
    n_classes = 700  # Number of classes in pretraining
    n_input_channels = 3  # Number of input channels (e.g., RGB)
    resnet_shortcut = 'B'
    conv1_t_size = 7
    conv1_t_stride = 1
    no_max_pool = False
    resnet_widen_factor = 1.0

opt = Opt()



# Generate the model
model = generate_model(opt)

print(model)

# Load the pretrained weights and set the final layer for fine-tuning
pretrain_path = 'r3d34_K_200ep.pth'
model_name = 'resnet'
n_finetune_classes = 700  # Set to your target number of classes



model = load_pretrained_model(model, pretrain_path, model_name, n_finetune_classes)

print(model)

ResNet(
  (conv1): Conv3d(3, 64, kernel_size=(7, 7, 7), stride=(1, 2, 2), padding=(3, 3, 3), bias=False)
  (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn2): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

/home/a30/ai_t4/namdt/violence_detection/resnet3d/model.py:100: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrain = torch.load(pretrain_path, map_location='cpu')


ResNet(
  (conv1): Conv3d(3, 64, kernel_size=(7, 7, 7), stride=(1, 2, 2), padding=(3, 3, 3), bias=False)
  (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn2): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [28]:
import torch.nn as nn
from resnet3d.models.resnet import BasicBlock

# Remove the last two layers by replacing them with nn.Identity
model.avgpool = nn.Identity()
model.fc = nn.Identity()

In [29]:
# Use a small subset of the train_loader for testing
num_test_batches = 2  # Number of batches to use for quick test
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [47]:
model.layer5 = nn.Sequential(
    # First block handles potential downsampling
        # First block handles potential downsampling
    BasicBlock(
        in_planes=512,
        planes=512,
        stride=2,  # Downsample spatially
        downsample=nn.Sequential(
            nn.Conv3d(512, 512, kernel_size=1, stride=2, bias=False),
            nn.BatchNorm3d(512)
        )
    ),
    # Additional blocks maintain dimensions
    BasicBlock(in_planes=512, planes=512),
    BasicBlock(in_planes=512, planes=512)
)

model.layer6 = nn.Sequential(
    # First block handles potential downsampling
        # First block handles potential downsampling
    BasicBlock(
        in_planes=512,
        planes=512,
        stride=2,  # Downsample spatially
        downsample=nn.Sequential(
            nn.Conv3d(512, 512, kernel_size=1, stride=2, bias=False),
            nn.BatchNorm3d(512)
        )
    ),
    # Additional blocks maintain dimensions
    BasicBlock(in_planes=512, planes=512),
    BasicBlock(in_planes=512, planes=512)
)


model.new_avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))

# Add binary classification layer (outputs logits for focal loss)
model.new_fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(512, 256),  # Intermediate layer
    nn.ReLU(inplace=False),  # Activation function
    nn.Dropout(0.5),      # Add dropout for regularization
    nn.Linear(256, 2)     # Binary classification (2 classes)
    # No activation function here - focal loss expects raw logits
)


original_forward = model.forward

# Define new forward method that includes the additional layers
def new_forward(self, x):
    print(f"Input shape: {x.shape}")  # Debugging line to check input tensor shape
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    if hasattr(self, 'maxpool'):
        x = self.maxpool(x)
        
    print("After layer 4 output shape:", x.shape)
    print("After layer 4 output requires_grad:", x.requires_grad)
    print(f"After layer 0: {x.shape}")  # Debugging line to check tensor shape after initial layers
    x = self.layer1(x)
    print(f"After layer 1: {x.shape}")  # Debugging line to check tensor shape after layer1
    x = self.layer2(x)
    print(f"After layer 2: {x.shape}")  # Debugging line to check tensor shape after layer2
    x = self.layer3(x)
    print(f"After layer 3: {x.shape}")  # Debugging line to check tensor shape after layer3
    x = self.layer4(x)
    print(f"After layer 4: {x.shape}")  # Debugging line to check tensor shape after layer4
    x = self.layer5(x)  # New layer5
    print(f"After layer 5: {x.shape}")  # Debugging line to check tensor shape after layer5
    x = self.layer6(x)  # New layer6
    print(f"After layer 6: {x.shape}")  # Debugging line to check tensor shape after layer6
    x = self.new_avgpool(x)
    print(f"After new_avgpool: {x.shape}")  # Debugging line to check tensor shape after new_avgpool    
    x = self.new_fc(x)
    print(f"After new_fc: {x.shape}")  # Debugging line to check tensor shape after new_fc
    return x

# Apply the new forward method
import types
model.forward = types.MethodType(new_forward, model)
model.to(device)

print("--- Parameters with requires_grad=True ---")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
# Print the modified model
# print(model)

--- Parameters with requires_grad=True ---
conv1.weight
bn1.weight
bn1.bias
layer1.0.conv1.weight
layer1.0.bn1.weight
layer1.0.bn1.bias
layer1.0.conv2.weight
layer1.0.bn2.weight
layer1.0.bn2.bias
layer1.1.conv1.weight
layer1.1.bn1.weight
layer1.1.bn1.bias
layer1.1.conv2.weight
layer1.1.bn2.weight
layer1.1.bn2.bias
layer1.2.conv1.weight
layer1.2.bn1.weight
layer1.2.bn1.bias
layer1.2.conv2.weight
layer1.2.bn2.weight
layer1.2.bn2.bias
layer2.0.conv1.weight
layer2.0.bn1.weight
layer2.0.bn1.bias
layer2.0.conv2.weight
layer2.0.bn2.weight
layer2.0.bn2.bias
layer2.0.downsample.0.weight
layer2.0.downsample.1.weight
layer2.0.downsample.1.bias
layer2.1.conv1.weight
layer2.1.bn1.weight
layer2.1.bn1.bias
layer2.1.conv2.weight
layer2.1.bn2.weight
layer2.1.bn2.bias
layer2.2.conv1.weight
layer2.2.bn1.weight
layer2.2.bn1.bias
layer2.2.conv2.weight
layer2.2.bn2.weight
layer2.2.bn2.bias
layer2.3.conv1.weight
layer2.3.bn1.weight
layer2.3.bn1.bias
layer2.3.conv2.weight
layer2.3.bn2.weight
layer2.3.bn2.bias

# TRAINING

In [39]:
from optflowdataset import create_data_loaders

# Data parameters
data_dir = "data"
dataset_name = "testdataset"  # Path: data/testdataset with violence and nonviolence subfolders
batch_size = 2
frame_size = 256
num_frames = 150

# # Create data loaders
# train_loader, val_loader, test_loader = create_data_loaders(
#     data_dir=data_dir,
#     dataset_name=dataset_name,
#     batch_size=batch_size,
#     frame_size=frame_size,
#     num_frames=num_frames,
#     train_split=0.7,
#     val_split=0.1,
#     num_workers=0,  # Set to 0 to avoid CUDA initialization issues
#     split_file=f"data/{dataset_name}_split.csv"  # Save splits for reproducibility
# )

# # Display dataset information
# print(f"Dataset loaded with frame size {frame_size}x{frame_size}, {num_frames} frames per video")
# print(f"Batch size: {batch_size}")

# # Verify input shape matches model expectations
# for batch_idx, (sequences, labels) in enumerate(train_loader):
#     print(f"Input shape: {sequences.shape}")  # Should be [batch_size, 3, 150, 512, 512]
#     print(f"Labels: {labels}")
#     output = model(sequences)  # Forward pass
#     print(f"Output shape: {output.shape}")  # Should be [batch_size,
#     break  # Just check the first batch

In [35]:
# Demonstrating the one-hot encoding option
train_loader_one_hot, val_loader_one_hot, test_loader_one_hot = create_data_loaders(
    data_dir=data_dir,
    dataset_name=dataset_name,
    batch_size=batch_size,
    frame_size=frame_size,
    num_frames=num_frames,
    num_workers=0,
    one_hot=True  # Enable one-hot encoding
)

# Check the first batch to see the one-hot encoded labels
for batch_idx, (sequences, labels) in enumerate(train_loader_one_hot):
    print(f"One-hot encoding example:")
    print(f"Input shape: {sequences.shape}")
    print(f"Labels shape: {labels.shape}")  # Should be [batch_size, 2]
    print(f"Labels: {labels}")
    break  # Just check the first batch

Loaded 1976 videos from testdataset
Nonviolence: 1000, Violence: 976
Loading dataset split from testdataset_split.csv
Dataset splits - Train: 1383, Val: 197, Test: 396
One-hot encoding example:
Input shape: torch.Size([2, 3, 150, 256, 256])
Labels shape: torch.Size([2, 2])
Labels: tensor([[0., 1.],
        [0., 1.]])


In [46]:
# Replace your existing training cell with this one:

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

# Use a small subset of the train_loader for testing
num_test_batches = 2  # Number of batches to use for quick test
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss and optimizer
# criterion = focal_loss_with_logits
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Alternative approach with standard BCEWithLogitsLoss

criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model.train()
print("Training with BCEWithLogitsLoss:")
for batch_idx, (inputs, targets) in enumerate(train_loader_one_hot):
    if batch_idx >= 1:
        break
        
    inputs = inputs.to(device)
    targets = targets.to(device).float()  # Make sure targets are float
    
    optimizer.zero_grad()
    outputs = model(inputs)
    
    loss = criterion(outputs, targets)
    
    # Print debugging info
    print(f"Output shape: {outputs.shape}, requires_grad: {outputs.requires_grad}")
    print(f"Loss value: {loss.item()}, requires_grad: {loss.requires_grad}")
    
    loss.backward()
    optimizer.step()
    print(f"Batch {batch_idx+1}, Loss: {loss.item():.4f}")

Training with BCEWithLogitsLoss:
Input shape: torch.Size([2, 3, 150, 256, 256])
After layer 4 output shape: torch.Size([2, 3, 150, 256, 256])
After layer 4 output requires_grad: False
After layer 0: torch.Size([2, 64, 75, 64, 64])
After layer 1: torch.Size([2, 64, 75, 64, 64])
After layer 2: torch.Size([2, 128, 38, 32, 32])
After layer 3: torch.Size([2, 256, 19, 16, 16])
After layer 4: torch.Size([2, 512, 10, 8, 8])
After layer 5: torch.Size([2, 512, 5, 4, 4])
After layer 6: torch.Size([2, 512, 3, 2, 2])
After new_avgpool: torch.Size([2, 512, 1, 1, 1])
After new_fc: torch.Size([2, 2])
Output shape: torch.Size([2, 2]), requires_grad: False
Loss value: 0.6600308418273926, requires_grad: False


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [48]:
# Check gradients for all layers and parameters

# First, ensure the model is in training mode
model.train()

# Get a single batch of data
for batch_idx, (inputs, targets) in enumerate(train_loader_one_hot):
    inputs = inputs.to(device)
    targets = targets.to(device).float()
    break

# Zero gradients
optimizer.zero_grad()

# Forward pass
outputs = model(inputs)
print(f"Output requires_grad: {outputs.requires_grad}")

# Use BCEWithLogitsLoss which works better with one-hot targets
loss_fn = nn.BCEWithLogitsLoss()
loss = loss_fn(outputs, targets)
print(f"Loss value: {loss.item()}, requires_grad: {loss.requires_grad}")

try:
    # Try backward pass
    loss.backward()
    print("Backward pass successful!")
except Exception as e:
    print(f"Backward pass failed with error: {e}")

# Check gradients for all parameters
print("\n--- Gradient Check for All Parameters ---")
for name, param in model.named_parameters():
    if param.requires_grad:
        if param.grad is not None:
            print(f"{name}: has gradients with norm {param.grad.norm().item()}")
        else:
            print(f"{name}: NO GRADIENTS")
    else:
        print(f"{name}: requires_grad=False")

# Check if any layer has inplace operations that might destroy gradient info
print("\n--- Checking for Inplace Operations ---")
for name, module in model.named_modules():
    if isinstance(module, nn.ReLU) and module.inplace:
        print(f"Warning: {name} uses inplace ReLU which can destroy gradient information")

# Check if the model was accidentally put in eval mode
print("\n--- Model Training Mode ---")
print(f"model.training: {model.training}")

# Check input gradient tracking
print("\n--- Input Gradient Tracking ---")
inputs.requires_grad = True
outputs_new = model(inputs)
print(f"When input requires_grad=True, output requires_grad: {outputs_new.requires_grad}")

Input shape: torch.Size([2, 3, 150, 256, 256])
After layer 4 output shape: torch.Size([2, 64, 75, 64, 64])
After layer 4 output requires_grad: False
After layer 0: torch.Size([2, 64, 75, 64, 64])
After layer 1: torch.Size([2, 64, 75, 64, 64])
After layer 2: torch.Size([2, 128, 38, 32, 32])
After layer 3: torch.Size([2, 256, 19, 16, 16])
After layer 4: torch.Size([2, 512, 10, 8, 8])
After layer 5: torch.Size([2, 512, 5, 4, 4])
After layer 6: torch.Size([2, 512, 3, 2, 2])
After new_avgpool: torch.Size([2, 512, 1, 1, 1])
After new_fc: torch.Size([2, 2])
Output requires_grad: False
Loss value: 0.6390318870544434, requires_grad: False
Backward pass failed with error: element 0 of tensors does not require grad and does not have a grad_fn

--- Gradient Check for All Parameters ---
conv1.weight: NO GRADIENTS
bn1.weight: NO GRADIENTS
bn1.bias: NO GRADIENTS
layer1.0.conv1.weight: NO GRADIENTS
layer1.0.bn1.weight: NO GRADIENTS
layer1.0.bn1.bias: NO GRADIENTS
layer1.0.conv2.weight: NO GRADIENTS
la